# Total Return Swap Pricing — Lou (2018)

**Reference:** W. Lou, *Pricing Total Return Swap*, 2018. SSRN 3217420.

Visual validation of equity TRS with repo financing and FVA.

## Contents
1. FVA as function of repo spread
2. Full-CSA TRS value vs spot
3. ATI funding spread decomposition
4. Bond forward under repo with haircut

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "python"))

import math
import numpy as np
import matplotlib.pyplot as plt
from pricebook.trs_lou import trs_equity_full_csa, trs_fva, trs_bond_forward

S0 = 100.0; r = 0.10; T = 1.0
D = math.exp(-r * T)
libor = (1/D - 1) / T  # ATI simply-compounded Libor

plt.rcParams.update({"figure.figsize": (10, 6), "font.size": 12})
print(f"S0={S0}, r={r}, D={D:.4f}, Libor(sc)={libor:.4f}")

## 1. FVA vs Repo Spread

$\text{fva} = (e^{(r_s - r)T} - 1) \times S_t \geq 0$

The hedge financing cost grows exponentially with the repo-OIS spread.

In [ ]:
spreads = np.linspace(0, 0.10, 100)
fvas = [trs_fva(S0, rs, T) for rs in spreads]

fig, ax = plt.subplots()
ax.plot(spreads * 10000, fvas, lw=2)
ax.axhline(0, color="k", lw=0.5)
ax.set_xlabel("Repo-OIS Spread (bp)")
ax.set_ylabel("FVA ($)")
ax.set_title(f"Hedge Financing Cost (Eq 8): S={S0}, T={T}Y")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 2. TRS Value vs Spot — Pre-Crisis vs Repo-Adjusted

Pre-crisis (Eq 2) vs full-CSA with repo (Eq 7). The repo-adjusted value is lower by the fva.

In [ ]:
spots = np.linspace(70, 130, 100)
sf = 0.02; r_f = libor + sf

v_precrisis = [(S0 * r_f * T + S0) * D - s for s in spots]
v_repo = [trs_equity_full_csa(s, S0, r_f, T, 0.0, D, rs_minus_r=0.05).value for s in spots]

fig, ax = plt.subplots()
ax.plot(spots, v_precrisis, lw=2, label="Pre-crisis (Eq 2, $r_s = r$)")
ax.plot(spots, v_repo, "--", lw=2, label="Repo-adjusted (Eq 7, $r_s - r$ = 500bp)")
ax.axhline(0, color="k", lw=0.5)
ax.axvline(S0, ls=":", color="gray", alpha=0.5, label=f"$S_0$ = {S0}")
ax.set_xlabel("Current Spot $S_t$")
ax.set_ylabel("TRS Value (payer)")
ax.set_title("Pre-Crisis vs Repo-Adjusted TRS Valuation")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Bond Forward under Repo with Haircut

$F = B_t e^{\bar{r}_s T} - \sum c_i e^{\bar{r}_s(T - T_i)}$ where $\bar{r}_s = (1-h)r_s + h r_N$

In [ ]:
B_t = 98.0
coupons = [(0.5, 2.5)]  # semi-annual 5% coupon on 100 face

haircuts = np.linspace(0, 0.30, 50)
rs = 0.03; rN = 0.05
forwards = []
for h in haircuts:
    rs_bar = (1 - h) * rs + h * rN
    fwd = trs_bond_forward(B_t, rs_bar, T=1.0, coupons=coupons, lambda_val=0.0)
    forwards.append(fwd)

fig, ax = plt.subplots()
ax.plot(haircuts * 100, forwards, lw=2)
ax.axhline(B_t, ls=":", color="gray", alpha=0.5, label=f"Spot = {B_t}")
ax.set_xlabel("Repo Haircut (%)")
ax.set_ylabel("Bond Forward Price")
ax.set_title(f"Bond Forward vs Haircut ($r_s$={rs*100:.0f}%, $r_N$={rN*100:.0f}%)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()